<a href="https://colab.research.google.com/github/brechman-ai/agentic-real-estate-chatbot/blob/main/realestate_chatbot_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install PDF dependency

!pip install pypdf openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.0/329.0 kB 6.8 MB/s eta 0:00:00


In [2]:
# Load OpenAI Client(using config.json)

import json
from openai import OpenAI

# Load config.json
with open("config.json", "r") as f:
    config = json.load(f)

# Extract values
api_key = config["API_KEY"]
api_base = config["OPENAI_API_BASE"]

# Create OpenAI client with explicit base URL
client = OpenAI(
    api_key=api_key,
    base_url=api_base
)

print("OpenAI client initialized with custom base URL ✅")




OpenAI client initialized with custom base URL ✅


In [3]:
#load and read the PDFs

import os
from pypdf import PdfReader

print("Files in workspace:", os.listdir())

def read_pdf(path):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() for page in reader.pages)

listings_text = read_pdf("unrealistic_real_estate_listings.pdf")
contracts_text = read_pdf("unrealistic_real_estate_contracts.pdf")

print("PDFs loaded successfully ✅")



Files in workspace: ['.config', 'unrealistic_real_estate_contracts.pdf', 'unrealistic_real_estate_listings.pdf', 'config.json', 'sample_data']
PDFs loaded successfully ✅


At this point:

1) Your chatbot knows the listings

2) Your chatbot knows the contracts

3) No embeddings yet → no errors

In [4]:
# run the PDF load cell

from pypdf import PdfReader

def read_pdf(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

listings_text = read_pdf("unrealistic_real_estate_listings.pdf")
contracts_text = read_pdf("unrealistic_real_estate_contracts.pdf")

combined_knowledge = listings_text + "\n\n" + contracts_text

print("combined_knowledge created ✅")


combined_knowledge created ✅


In [5]:
# chunk the pdf text

def split_text(text, chunk_size=800):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i + chunk_size]))
    return chunks

pdf_chunks = split_text(combined_knowledge)

print(f"Created {len(pdf_chunks)} text chunks ✅")



Created 1 text chunks ✅


In [6]:
# simple context selection (no Embeddings). For demo purposes, we'll just pass the most relevant looking chunk by keyword matching
#this is stable and demo-safe

def find_relevant_chunk(question):
    for chunk in pdf_chunks:
        if any(word.lower() in chunk.lower() for word in question.split()):
            return chunk
    return pdf_chunks[0]


In [7]:
#Chatbot Using PDF knowledge
def chatbot(question):
    context = find_relevant_chunk(question)

    prompt = f"""
You are a real estate assistant.

Answer the question using ONLY the information below.
If the answer is not in the document, say "I don't know based on the documents."

DOCUMENT:
{context}

QUESTION:
{question}

ANSWER:
"""

    response = client.responses.create(
        model="gpt-4o-mini", #i switch from gpt-4.1-mini to gpt-4.o mini to avoid error to run the following codes for the questions
        input=prompt
    )

    return response.output_text.strip()




In [ ]:
#test wwith PDF-Based Questions. Run these one by one

In [8]:
print(chatbot("What unrealistic listings do you have?"))


The unrealistic listings are:

1. Oceanfront Mansion in Arizona - Price: $150,000
   Description: A 12-bedroom oceanfront mansion located in the Arizona desert, featuring private waves, indoor rainforest, and a rooftop ski slope.

2. Floating Glass Villa in Downtown Manhattan - Price: $300,000
   Description: A fully floating glass home hovering 20 feet above street level with drone valet parking and AI-managed sunlight control.

3. Underground Luxury Bunker Estate - Price: $95,000
   Description: A 20,000 sq ft underground estate with artificial sky ceilings, private subway access, and geothermal champagne cellar.


In [9]:
print(chatbot("Tell me about the intergalactic property contract"))


The Intergalactic Property Purchase Agreement involves a buyer agreeing to purchase a lunar-adjacent property pending approval from Earth and Mars authorities, with the closing date scheduled upon successful teleportation of funds.


In [10]:
print(chatbot("Is there a floating villa in Manhattan?"))


Yes, there is a floating glass villa in Downtown Manhattan.


In [11]:
print(chatbot("Do you sell normal suburban homes?"))


#this one should trigger a "not in documents" style response - that's a trust signal in demos

I don't know based on the documents.


Let's test the agent to see if it was all luck and not robust by runining:
validation, evaluation and justification

In [12]:
#Validation and stress-test the pipeline(most important)

test_questions = [
    "What unrealistic listings do you have?",
    "List properties that violate basic real estate laws",
    "Do any listings conflict with the contracts?",
    "Are there any unicorn-related properties?",
    "Summarize a contract that is not in the documents",
    "Who is the president of the United States?"
]

def run_tests(chatbot_fn, questions):
    results = []
    for q in questions:
        try:
            answer = chatbot_fn(q)
            results.append({
                "question": q,
                "answer": answer[:300]
            })
        except Exception as e:
            results.append({
                "question": q,
                "answer": f"ERROR: {str(e)}"
            })
    return results

test_results = run_tests(chatbot, test_questions)

for r in test_results:
    print("\nQ:", r["question"])
    print("A:", r["answer"])



Q: What unrealistic listings do you have?
A: The unrealistic listings are:

1. Oceanfront Mansion in Arizona - Price: $150,000
   Description: A 12-bedroom oceanfront mansion located in the Arizona desert, featuring private waves, indoor rainforest, and a rooftop ski slope.

2. Floating Glass Villa in Downtown Manhattan - Price: $300,000
   De

Q: List properties that violate basic real estate laws
A: 1. Oceanfront Mansion in Arizona
2. Floating Glass Villa in Downtown Manhattan
3. Underground Luxury Bunker Estate
4. Intergalactic Property Purchase Agreement
5. Time-Travel Property Lease
6. AI-Owned Property Transfer Agreement

Q: Do any listings conflict with the contracts?
A: I don't know based on the documents.

Q: Are there any unicorn-related properties?
A: I don't know based on the documents.

Q: Summarize a contract that is not in the documents
A: I don't know based on the documents.

Q: Who is the president of the United States?
A: I don't know based on the documents.


In [13]:
#add evaluation results.
#step 1- evaluation function

def evaluate_response(question, answer, source_text):
    score = {
        "grounded": 0,
        "abstained_correctly": 0,
        "useful": 0
    }

    # Check abstention
    abstain_phrases = [
        "not in the document",
        "not found in the provided",
        "i don't know",
        "no information available"
    ]

    if any(p in answer.lower() for p in abstain_phrases):
        score["abstained_correctly"] = 1

    # Groundedness: overlap with source text
    overlap = sum(1 for word in answer.split() if word.lower() in source_text.lower())
    if overlap > 10:
        score["grounded"] = 1

    # Usefulness
    if len(answer.strip()) > 50:
        score["useful"] = 1

    final_score = sum(score.values())
    return final_score, score




In [14]:
#step2- run evaluation on test questions
evaluation_results = []

for q in test_questions:
    answer = chatbot(q)
    score, breakdown = evaluate_response(q, answer, combined_knowledge)

    evaluation_results.append({
        "question": q,
        "answer": answer[:200],
        "score": score,
        "details": breakdown
    })

for r in evaluation_results:
    print("\nQ:", r["question"])
    print("Score:", r["score"], "/ 3")
    print("Details:", r["details"])




Q: What unrealistic listings do you have?
Score: 2 / 3
Details: {'grounded': 1, 'abstained_correctly': 0, 'useful': 1}

Q: List properties that violate basic real estate laws
Score: 2 / 3
Details: {'grounded': 1, 'abstained_correctly': 0, 'useful': 1}

Q: Do any listings conflict with the contracts?
Score: 1 / 3
Details: {'grounded': 0, 'abstained_correctly': 1, 'useful': 0}

Q: Are there any unicorn-related properties?
Score: 1 / 3
Details: {'grounded': 0, 'abstained_correctly': 1, 'useful': 0}

Q: Summarize a contract that is not in the documents
Score: 1 / 3
Details: {'grounded': 0, 'abstained_correctly': 1, 'useful': 0}

Q: Who is the president of the United States?
Score: 1 / 3
Details: {'grounded': 0, 'abstained_correctly': 1, 'useful': 0}


In [15]:
#Step3- Aggregate Metrics

avg_score = sum(r["score"] for r in evaluation_results) / len(evaluation_results)

print(f"\n📊 Average Evaluation Score: {avg_score:.2f} / 3")



📊 Average Evaluation Score: 1.33 / 3


In [16]:
# Part 2 Feedbacks loop(agent self improvement)
#feedback Memory store

feedback_memory = {
    "failed_questions": [],
    "successful_questions": []
}


In [17]:
#log performance Automatically

def log_feedback(question, answer, score):
    if score < 2:
        feedback_memory["failed_questions"].append({
            "question": question,
            "answer": answer
        })
    else:
        feedback_memory["successful_questions"].append(question)


In [18]:
#integrate it
for q in test_questions:
    answer = chatbot(q)
    score, _ = evaluate_response(q, answer, combined_knowledge)
    log_feedback(q, answer, score)



In [19]:
#part3- prompt self-correction(real agent loop)

#Adaptive system prompt

def build_system_prompt():
    base_prompt = """
You are a real estate assistant.
Answer ONLY using the provided documents.
If information is missing, say so clearly.
"""

    if len(feedback_memory["failed_questions"]) > 0:
        base_prompt += "\nBe extra cautious about hallucinations and missing data."

    return base_prompt


In [20]:
#update your chatbot function:
def chatbot(user_question):
    system_prompt = build_system_prompt()
    context = retrieve_context(user_question)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {user_question}"}
        ]
    )

    return response.choices[0].message.content



In [21]:
#Show the feedback loop working important

print("❌ Failed Questions:")
for f in feedback_memory["failed_questions"]:
    print("-", f["question"])

print("\n✅ Successful Questions:")
for s in feedback_memory["successful_questions"]:
    print("-", s)



❌ Failed Questions:
- Do any listings conflict with the contracts?
- Are there any unicorn-related properties?
- Summarize a contract that is not in the documents
- Who is the president of the United States?

✅ Successful Questions:
- What unrealistic listings do you have?
- List properties that violate basic real estate laws


In [22]:
#LLM as a judge(evaluator agent)

EVALUATOR_PROMPT = """
You are an impartial evaluator for a real estate document QA system.

Your task:
- Judge whether the assistant answer is grounded ONLY in the provided context.
- Penalize hallucinations.
- Reward correct abstention.

Return your evaluation strictly in JSON with the following keys:
- grounded (true/false)
- hallucinated (true/false)
- abstained_correctly (true/false)
- score (0 to 5)
- explanation (short sentence)
"""


In [23]:
#Evaluator function (llm Judge)

def llm_judge(question, answer, context):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": EVALUATOR_PROMPT},
            {"role": "user", "content": f"""
Question:
{question}

Context:
{context}

Assistant Answer:
{answer}
"""}
        ],
        temperature=0
    )

    return response.choices[0].message.content


In [ ]:
#structure evaluation pipeline

In [27]:
#let's retrive_context minimal+safe

def retrieve_context(query, top_k=3):
    """
    Simple keyword-based retrieval.
    Returns the most relevant chunks.
    """

    query_words = set(query.lower().split())
    scored_chunks = []

    for chunk in pdf_chunks:
        chunk_words = set(chunk.lower().split())
        score = len(query_words.intersection(chunk_words))
        scored_chunks.append((score, chunk))

    scored_chunks.sort(reverse=True, key=lambda x: x[0])

    top_chunks = [chunk for score, chunk in scored_chunks[:top_k] if score > 0]

    if not top_chunks:
        return "No relevant context found in documents."

    return "\n\n".join(top_chunks)


In [28]:
#run the judge on the test set
import json

judged_results = []

for q in test_questions:
    answer = chatbot(q)
    context = retrieve_context(q)

    judge_output = llm_judge(q, answer, context)

    try:
        evaluation = json.loads(judge_output)
    except:
        evaluation = {
            "grounded": False,
            "hallucinated": True,
            "abstained_correctly": False,
            "score": 0,
            "explanation": "Evaluator output malformed"
        }

    judged_results.append({
        "question": q,
        "answer": answer[:200],
        "evaluation": evaluation
    })



In [25]:
#Display Results(notebook-friendly)

for r in judged_results:
    print("\nQ:", r["question"])
    print("Score:", r["evaluation"]["score"], "/ 5")
    print("Grounded:", r["evaluation"]["grounded"])
    print("Hallucinated:", r["evaluation"]["hallucinated"])
    print("Explanation:", r["evaluation"]["explanation"])


In [29]:
#Aggregate Metrics
avg_score = sum(r["evaluation"]["score"] for r in judged_results) / len(judged_results)
hallucination_rate = sum(1 for r in judged_results if r["evaluation"]["hallucinated"]) / len(judged_results)

print(f"\n📊 Average Judge Score: {avg_score:.2f} / 5")
print(f"🚨 Hallucination Rate: {hallucination_rate:.2%}")




📊 Average Judge Score: 4.67 / 5
🚨 Hallucination Rate: 0.00%


In [30]:
#Self- correcting Agent Loop

def stricter_prompt():
    return """
IMPORTANT:
- Do NOT guess.
- Do NOT invent facts.
- If the answer is not in the documents, say:
  'This information is not available in the provided documents.'
"""


In [31]:
#agent with self correction

def agentic_chatbot(question, max_retries=1):
    answer = chatbot(question)
    context = retrieve_context(question)

    judge = llm_judge(question, answer, context)
    evaluation = json.loads(judge)

    if evaluation["score"] < 3 or evaluation["hallucinated"]:
        if max_retries > 0:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": stricter_prompt()},
                    {"role": "user", "content": f"Context:\n{context}\n\nQuestion:\n{question}"}
                ],
                temperature=0
            )
            return response.choices[0].message.content

    return answer


In [32]:
#Final test
print(agentic_chatbot("Summarize a contract not in the documents"))


The request for summarizing a contract not in the provided documents cannot be fulfilled as there is no available information on such a contract.
